# Glossary Formatter

This notebook parses `data/glossary.md` into a consolidated glossary table and computes in-notebook summaries/visualizations.

Exports:
- `outputs/glossary/glossary.tsv`

---

## New Implementation — Parsing `glossary.md`

Parses `data/glossary.md` and writes a single consolidated TSV:

- `outputs/glossary/glossary.tsv`

Summary statistics and the example-share chart are computed below without writing additional files.


In [ ]:
from __future__ import annotations

import re
from pathlib import Path
from urllib.parse import unquote

import matplotlib.pyplot as plt
import pandas as pd

WORKDIR = Path.cwd()
if not (WORKDIR / 'data').exists():
    WORKDIR = WORKDIR.parent
GLOSSARY_MD_PATH = WORKDIR / 'data' / 'glossary.md'
OUTPUT_DIR = WORKDIR / 'outputs' / 'glossary'
OUTPUT_TSV = OUTPUT_DIR / 'glossary.tsv'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GLOSSARY_MD_PATH


In [ ]:
def _extract_link_url(text: str) -> str:
    m = re.search(r'\]\((https?://[^)]+)\)', text)
    return unquote(m.group(1).strip()) if m else ''


def _parse_label(block: str, label: str) -> str:
    m = re.search(rf'_{re.escape(label)}_:\s*(.*?)(?=\s*_[^_]+_:|$)', block, re.S)
    if not m:
        return ''
    return re.sub(r'\s+', ' ', m.group(1)).strip()


def parse_glossary_md(path: Path) -> pd.DataFrame:
    raw = path.read_text(encoding='utf-8', errors='ignore')
    # Normalize backslash line-continuations (soft-wrap artifacts from HTML source)
    raw = re.sub(r'\\\n\s*', '\n', raw)

    # Split into per-term blocks at each **term** heading
    entries = re.split(r'\n(?=\*\*)', raw)

    rows: list[dict] = []
    for entry in entries:
        term_m = re.match(r'\*\*([^*]+)\*\*', entry.strip())
        if not term_m:
            continue
        term = term_m.group(1).strip()

        sf_m = re.search(r'_Surface forms_:\s*([^\n]+)', entry)
        surface_forms = sf_m.group(1).strip() if sf_m else ''

        persona_block_m = re.search(
            r'_Persona/In-Group_:.*?(?=\nDescription|\nExample|\Z)', entry, re.S
        )
        persona_block = persona_block_m.group(0) if persona_block_m else ''
        persona = _parse_label(persona_block, 'Persona/In-Group')
        covert_meaning = _parse_label(persona_block, 'Covert (in-group) meaning')
        dogwhistle_type = _parse_label(persona_block, 'Type')
        register = _parse_label(persona_block, 'Register')

        desc_header_m = re.search(r'Description \(from [^\n]+', entry)
        description_source = _extract_link_url(desc_header_m.group(0)) if desc_header_m else ''
        desc_body_m = re.search(
            r'Description \(from [^\n]+\n(.*?)(?=\nExample context|\Z)', entry, re.S
        )
        description = (
            re.sub(r'\s+', ' ', desc_body_m.group(1)).strip() if desc_body_m else ''
        )

        examples, ex_sources, ex_speakers, ex_dates = [], [], [], []
        for ex_m in re.finditer(
            r'Example context \(in ([^\n]+)\n(.*?)(?=\nExample context|\Z)', entry, re.S
        ):
            source_url = _extract_link_url(ex_m.group(1))
            body = ex_m.group(2)
            speaker = _parse_label(body, 'Speaker')
            date = _parse_label(body, 'Date')
            ex_text = re.sub(r'_Speaker_:.*', '', body, flags=re.S).strip()
            ex_text = re.sub(r'\s+', ' ', ex_text).strip()
            if ex_text:
                examples.append(ex_text)
                ex_sources.append(source_url)
                ex_speakers.append(speaker)
                ex_dates.append(date)

        rows.append(
            {
                'term': term,
                'surface_forms': surface_forms,
                'persona_in_group': persona,
                'covert_meaning': covert_meaning,
                'type': dogwhistle_type,
                'register': register,
                'description': description,
                'description_source': description_source,
                'example_count': len(examples),
                'examples': ' || '.join(examples),
                'example_sources': ' ; '.join(s for s in ex_sources if s),
                'example_speakers': ' ; '.join(s for s in ex_speakers if s),
                'example_dates': ' ; '.join(s for s in ex_dates if s),
            }
        )

    return pd.DataFrame(rows)


glossary_df = parse_glossary_md(GLOSSARY_MD_PATH)
glossary_df.to_csv(OUTPUT_TSV, sep='\t', index=False)

print(f'Wrote {len(glossary_df):,} term rows  →  {OUTPUT_TSV}')
glossary_df.head(5)


In [ ]:
# --- Group metrics summary ---
group_cols = ['persona_in_group', 'type', 'register']
group_metrics = (
    glossary_df.groupby(group_cols, dropna=False, as_index=False)
    .agg(
        term_count=('term', 'count'),
        unique_term_count=('term', 'nunique'),
        total_examples=('example_count', 'sum'),
        avg_examples_per_term=('example_count', 'mean'),
    )
    .sort_values(['total_examples', 'term_count'], ascending=False)
)
group_metrics['avg_examples_per_term'] = group_metrics['avg_examples_per_term'].round(3)

print(f'{len(group_metrics):,} persona/type/register groups')
group_metrics.head(15)


In [ ]:
# --- Example-share pie chart by persona/in-group ---
examples_by_group = (
    glossary_df.groupby('persona_in_group', dropna=False)['example_count']
    .sum()
    .sort_values(ascending=False)
)

examples_by_group.index = [
    idx if isinstance(idx, str) and idx.strip() else 'Unknown'
    for idx in examples_by_group.index
]

plt.figure(figsize=(9, 9))
plt.pie(
    examples_by_group.values,
    labels=examples_by_group.index,
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.8,
)
plt.title('Share of Example Contexts by Persona/In-Group')
plt.tight_layout()
plt.show()

examples_by_group.to_frame('example_count')
